<a href="https://colab.research.google.com/github/Sezikeye/Assignment-AI/blob/main/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from sklearn.datasets import fetch_20newsgroups

# Load the full dataset (train split by default)
newsgroups_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)

# Load the test set
newsgroups_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

# Explore basic structure
print(newsgroups_train.keys())

# Example: categories
print(newsgroups_train.target_names)

# Example: a single document
print(newsgroups_train.data[0])

# Example: label of first document
print(newsgroups_train.target[0])

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of prod

In [9]:
# we install and import preprossessing tool
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
#Define preprocessing function
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove punctuation and non-letters
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

In [13]:
texts = newsgroups_train.data
labels = newsgroups_train.target

In [15]:
# we implement Bag of Words and TF-IDF for feature extraction.
from sklearn.feature_extraction.text import CountVectorizer

# Initialize BoW model
bow_vectorizer = CountVectorizer(
    stop_words='english',   # removes common English stopwords
    max_features=5000       # limit vocabulary size for efficiency
)

# Fit and transform
X_bow = bow_vectorizer.fit_transform(texts)

print("BoW shape:", X_bow.shape)

BoW shape: (11314, 5000)


In [16]:
#TF-IDF (Term Frequency–Inverse Document Frequency)
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF model
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

# Fit and transform
X_tfidf = tfidf_vectorizer.fit_transform(texts)

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (11314, 5000)


In [17]:
# we Compare BoW vs TF-IDF
print("BoW sample row (non-zero values):")
print(X_bow[0])

print("\nTF-IDF sample row (non-zero values):")
print(X_tfidf[0])

BoW sample row (non-zero values):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 45 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 4826)	2
  (0, 4655)	2
  (0, 1593)	2
  (0, 4499)	1
  (0, 4330)	1
  (0, 892)	5
  (0, 3111)	1
  (0, 3474)	1
  (0, 2229)	1
  (0, 3239)	1
  (0, 4675)	1
  (0, 2822)	1
  (0, 1070)	1
  (0, 3303)	1
  (0, 2684)	1
  (0, 50)	1
  (0, 4917)	1
  (0, 3958)	1
  (0, 1343)	1
  (0, 1521)	1
  (0, 4230)	1
  (0, 2725)	1
  (0, 2596)	1
  (0, 1569)	1
  (0, 862)	1
  (0, 1522)	1
  (0, 3704)	1
  (0, 4158)	1
  (0, 346)	1
  (0, 4043)	1
  (0, 3823)	1
  (0, 756)	1
  (0, 2565)	1
  (0, 2959)	1
  (0, 1637)	1
  (0, 4215)	1
  (0, 4978)	1
  (0, 3552)	1
  (0, 2188)	1
  (0, 2355)	1
  (0, 2726)	1
  (0, 2775)	1
  (0, 4491)	1
  (0, 2292)	1
  (0, 808)	1

TF-IDF sample row (non-zero values):
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 45 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 4826)	0.36148489933726585
  (0, 4655)	0.29489583134583186
  (0, 1

In [22]:
#K-Means Clustering
from sklearn.cluster import KMeans

num_clusters = 20  # 20 Newsgroups categories (for reference)

kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(X_tfidf)

labels = kmeans.labels_

In [24]:
# we inspect clusters

# Show top terms per cluster
terms = tfidf_vectorizer.get_feature_names_out()

def print_top_terms(cluster_num, n_terms=10):
    center = kmeans.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-n_terms:][::-1]
    print(f"\nCluster {cluster_num}:")
    print([terms[i] for i in top_indices])

print_top_terms(0)
print_top_terms(1)


Cluster 0:
['israel', 'israeli', 'jews', 'arab', 'arabs', 'jake', 'edu', 'lebanese', 'israelis', 'adam']

Cluster 1:
['berkeley', 'wpi', '00', 'edu', 'posting', 'nntp', 'host', 'garnet', 'california', 'polytechnic']


In [27]:
X = tfidf_vectorizer.transform(texts)
kmeans.fit(X)

KMeans(n_clusters=20, n_init=10, random_state=42)

In [28]:
# we create unseen data
new_documents = [
    "NASA launched a new satellite into orbit around Mars",
    "My car engine makes a strange noise when I accelerate",
    "The government passed a new law about healthcare reform",
    "I am having trouble installing Windows on my laptop"
]

In [31]:
#Transform new documents using SAME vectorizer
X_new = tfidf_vectorizer.transform(new_documents)

In [32]:
# we predict clusters
cluster_predictions = kmeans.predict(X_new)

for doc, cluster in zip(new_documents, cluster_predictions):
    print(f"\nDocument: {doc}")
    print(f"Assigned Cluster: {cluster}")


Document: NASA launched a new satellite into orbit around Mars
Assigned Cluster: 14

Document: My car engine makes a strange noise when I accelerate
Assigned Cluster: 7

Document: The government passed a new law about healthcare reform
Assigned Cluster: 2

Document: I am having trouble installing Windows on my laptop
Assigned Cluster: 3


In [35]:
import scipy
import numpy as np

# Combine training + new data
X_combined = scipy.sparse.vstack([X, X_new])

labels_combined = np.concatenate([
    labels,          # training cluster labels
    cluster_predictions       # new document cluster labels
])

In [36]:
# we check for similarity
from sklearn.metrics import silhouette_score

score = silhouette_score(X_combined, labels_combined, metric='cosine')
print("Silhouette Score:", score)

Silhouette Score: 0.023873378114479604


In [37]:
# cohesion
from sklearn.metrics import pairwise_distances
import numpy as np

def cohesion(X, labels):
    unique_clusters = np.unique(labels)
    total = 0

    for c in unique_clusters:
        cluster_points = X[labels == c]
        if cluster_points.shape[0] > 1:
            distances = pairwise_distances(cluster_points, metric='cosine')
            total += np.mean(distances)

    return total / len(unique_clusters)

print("Cohesion:", cohesion(X_combined, labels_combined))

Cohesion: 0.8629806743078723


In [39]:
# separation distance between clusters
def separation(X, labels):
    centroids = []

    for c in np.unique(labels):
        cluster_points = X[labels == c]
        centroid = np.asarray(cluster_points.mean(axis=0)) # Convert np.matrix to np.ndarray
        centroids.append(centroid)

    centroids = np.vstack(centroids)
    distances = pairwise_distances(centroids, metric='cosine')

    # remove diagonal (self-distance)
    np.fill_diagonal(distances, np.nan)

    return np.nanmean(distances)

print("Separation:", separation(X_combined, labels_combined))

Separation: 0.7574767982729063
